In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.environ['MPLBACKEND'] = 'Agg'

In [3]:
%%bash
set -e
rm -rf /content/pyskl
git clone https://github.com/uub-buu/pyskl.git /content/pyskl

if [ ! -d /usr/local/miniconda ]; then
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
    bash miniconda.sh -b -p /usr/local/miniconda
fi

CONDA=/usr/local/miniconda/bin/conda
$CONDA tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true
$CONDA tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r || true
$CONDA create -y -n pyskl python=3.10

$CONDA run -n pyskl pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 \
    -f https://download.pytorch.org/whl/torch_stable.html
$CONDA run -n pyskl pip install "numpy<2" "setuptools<81" "yapf<0.40"
$CONDA run -n pyskl pip install mmcv-full==1.5.0 \
    -f https://download.openmmlab.com/mmcv/dist/cu113/torch1.11.0/index.html

cd /content/pyskl
$CONDA run -n pyskl pip install -e . --no-deps
$CONDA run -n pyskl pip install matplotlib scipy decord fvcore moviepy pymemcache opencv-contrib-python

echo "Setup complete."

PREFIX=/usr/local/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /usr/local/miniconda
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: ...working... done
Channels:
 - defaults
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /usr/local/miniconda/envs/pyskl

  added / updated specs:
    - python=3.10


The following pac

Cloning into '/content/pyskl'...


==> WARNING: A newer version of conda exists. <==
    current version: 26.7.1
    latest version: 26.7.2

Please update conda by running

    $ conda self update


WARNING conda.conda_pypi.main:notify_externally_managed_future(156): 
  Did you know? You can install many PyPI packages with conda
  using the conda-pypi beta. Get started:
    https://docs.conda.io/projects/conda/en/stable/new-features.html

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyskl 0.1.0 requires mmdet==2.23.0, which is not installed.
pyskl 0.1.0 requires mmpose==0.24.0, which is not installed.


In [5]:
!grep -n "einsum" /content/pyskl/pyskl/models/gcns/utils/gcn.py | grep -E "^(7[0-9]|8[0-9]):"

In [6]:
%%writefile /content/export_onnx.py
import os
os.environ['MPLBACKEND'] = 'Agg'

import torch
import torch.nn as nn
from mmcv import Config
from pyskl.models import build_model

cfg = Config.fromfile('/content/pyskl/configs/stgcn/stgcn_lite_hmdb51_hrnet/j.py')
model = build_model(cfg.model)

checkpoint = torch.load(
    '/content/drive/MyDrive/wes237b_project/work_dirs/lite_stgcn_hmdb51/best_top1_acc_epoch_14.pth',
    map_location='cpu'
)
model.load_state_dict(checkpoint['state_dict'])
model.eval()

class InferenceWrapper(nn.Module):
    def __init__(self, backbone, cls_head):
        super().__init__()
        self.backbone = backbone
        self.cls_head = cls_head

    def forward(self, keypoint):
        x = self.backbone(keypoint)
        return self.cls_head(x)

wrapper = InferenceWrapper(model.backbone, model.cls_head)
wrapper.eval()

dummy_input = torch.randn(1, 2, 100, 17, 3)

with torch.no_grad():
    out = wrapper(dummy_input)

print("Forward pass output shape:", out.shape)
assert out.shape == (1, 51), f"WRONG OUTPUT SHAPE: {out.shape} -- this is NOT a complete classifier export!"

torch.onnx.export(
    wrapper,
    dummy_input,
    '/content/drive/MyDrive/wes237b_project/lite_stgcn_hmdb51_matmul.onnx',
    input_names=['keypoint'],
    output_names=['action_scores'],
    opset_version=12,
    dynamic_axes=None
)
print("Export complete. Output shape confirmed:", out.shape)

Writing /content/export_onnx.py


In [11]:
!/usr/local/miniconda/bin/conda run -n pyskl pip install "numpy<2"

  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyskl 0.1.0 requires mmdet==2.23.0, which is not installed.
pyskl 0.1.0 requires mmpose==0.24.0, which is not installed.
opencv-contrib-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [12]:
!/usr/local/miniconda/bin/conda run -n pyskl python /content/export_onnx.py

Forward pass output shape: torch.Size([1, 51])
Export complete. Output shape confirmed: torch.Size([1, 51])
/usr/local/miniconda/envs/pyskl/lib/python3.10/site-packages/torch/torch_version.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]
/content/pyskl/pyskl/models/heads/simple_head.py:89: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert x.shape[1] == self.in_c


In [14]:
%%writefile /content/check_ops.py
import onnx
model = onnx.load('/content/drive/MyDrive/wes237b_project/lite_stgcn_hmdb51_matmul.onnx')
ops = set(node.op_type for node in model.graph.node)
print('Einsum present:', 'Einsum' in ops)
print('MatMul present:', 'MatMul' in ops)

Writing /content/check_ops.py


In [ ]:
!/usr/local/miniconda/bin/conda run -n pyskl pip install onnx

In [17]:
!/usr/local/miniconda/bin/conda run -n pyskl python /content/check_ops.py

  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 122.3 MB/s  0:00:00
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyskl 0.1.0 requires mmdet==2.23.0, which is not installed.
pyskl 0.1.0 requires mmpose==0.24.0, which is not installed.
Einsum present: False
MatMul present: True
